<a href="https://colab.research.google.com/github/Claud1601/CN7030-Assignment/blob/main/tutorial_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip3 install pyspark

In [ ]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Tutorial1_CN7030") \
                            .master("local[*]")\
                            .config("spark.some.config.option", "some-value") \
                            .getOrCreate()

Part 2: Create PySpark DataFrame with an explicit schema

In [ ]:
# PySpark DataFrame with Explicit Schema
df = spark.createDataFrame([
    (1, 87.0, 'Mike', 'math', 'passed'),
    (2, 60.5, 'Mike', 'computing', 'failed'),
    (3, 20.8, 'Mina', 'networking', 'passed'),
    (4, 41.0, 'Emmy', 'math', 'failed'),
    (5, 39.0, 'Alex', 'computing', 'failed'),
    (6, 55.8, 'Alex', 'AI', 'passed'),
    (7, 74.0, 'Emmy', 'AI', 'passed'),
    ], schema = 'ID int, Mark double, Name string, Lesson string, Status string')

# show table
df.show()

# show schema
df.printSchema()

+---+----+----+----------+------+
| ID|Mark|Name|    Lesson|Status|
+---+----+----+----------+------+
|  1|87.0|Mike|      math|passed|
|  2|60.5|Mike| computing|failed|
|  3|20.8|Mina|networking|passed|
|  4|41.0|Emmy|      math|failed|
|  5|39.0|Alex| computing|failed|
|  6|55.8|Alex|        AI|passed|
|  7|74.0|Emmy|        AI|passed|
+---+----+----+----------+------+

root
 |-- ID: integer (nullable = true)
 |-- Mark: double (nullable = true)
 |-- Name: string (nullable = true)
 |-- Lesson: string (nullable = true)
 |-- Status: string (nullable = true)



In [ ]:
data = [[295, "South Bend", "Indiana",  101190, 112.9]]
columns = ["rank", "city", "state",  "population", "price"]

df1 = spark.createDataFrame(data, schema="rank LONG, city STRING, state STRING,  population LONG, price DOUBLE")
display(df1)
df1.show()

DataFrame[rank: bigint, city: string, state: string, population: bigint, price: double]

+----+----------+-------+----------+-----+
|rank|      city|  state|population|price|
+----+----------+-------+----------+-----+
| 295|South Bend|Indiana|    101190|112.9|
+----+----------+-------+----------+-----+



Part 3: Create PySpark DataFrame from Pandas DataFrame

In [ ]:
import numpy as np
import pandas as pd
df_pandas = pd.DataFrame(np.random.randint(0,200,size=(100000, 5)), columns=list('ABCDE'))

In [ ]:
df2 = spark.createDataFrame(df_pandas)

# show table
df2.show()

# show schema
df2.printSchema()

+---+---+---+---+---+
|  A|  B|  C|  D|  E|
+---+---+---+---+---+
|188| 10|103|154| 27|
| 38| 26| 15|179|125|
|137|117|  7|162| 95|
|193| 19|128|151| 92|
| 10|  1|178| 15|  2|
| 70| 53| 11|171| 30|
| 25| 47|125| 58|177|
| 14| 63|  4|105| 68|
|140| 44|182|152| 15|
| 98| 96|108| 83|177|
|117|127| 92| 72| 58|
|  1|190|120| 83|152|
|128|191| 79|156|195|
|154| 51| 86| 53| 91|
|178|145| 95| 15| 92|
|153|159| 30|185|112|
| 10|170|137| 44| 87|
|198| 49| 43|161| 72|
| 16| 11|111|151| 82|
|186| 63|178|138|101|
+---+---+---+---+---+
only showing top 20 rows
root
 |-- A: long (nullable = true)
 |-- B: long (nullable = true)
 |-- C: long (nullable = true)
 |-- D: long (nullable = true)
 |-- E: long (nullable = true)



In [ ]:
print(df2.count())

100000


In [ ]:
# Rename columns of df1 to match df's column names
df1_renamed = df1.withColumnRenamed("rank", "ID") \
                 .withColumnRenamed("city", "Mark") \
                 .withColumnRenamed("state", "Name") \
                 .withColumnRenamed("population", "Lesson") \
                 .withColumnRenamed("price", "Status")

print("Schema of df1 after renaming:")
df1_renamed.printSchema()

print("\nOriginal schema of df:")
df.printSchema()

Schema of df1 after renaming:
root
 |-- ID: long (nullable = true)
 |-- Mark: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Lesson: long (nullable = true)
 |-- Status: double (nullable = true)


Original schema of df:
root
 |-- ID: integer (nullable = true)
 |-- Mark: double (nullable = true)
 |-- Name: string (nullable = true)
 |-- Lesson: string (nullable = true)
 |-- Status: string (nullable = true)



In [ ]:
Totpartition=df.rdd.getNumPartitions()
print(Totpartition)
newpartitiondf = df.repartition(4)

print(newpartitiondf.rdd.getNumPartitions())

# Repatition by column name into 4 partitions, check the df1 it has a column name city we want it to be in four paritition
df4 = df1.repartition(4, "city")
print(df4.rdd.getNumPartitions())
# Repatition by multiple columns
#df5 = df4.repartition("ColumnName1","ColumnName2")

2
4
4


Part 4: Create PySpark DataFrame from CSV


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving PlayersDataset.csv to PlayersDataset (1).csv


Once the file is uploaded, we can load it into the `df_fifa` DataFrame.

In [ ]:
file_name = list(uploaded.keys())[0]
print(f"File '{file_name}' uploaded successfully.")

df_fifa = spark.read.format("csv").load(file_name, inferSchema=True, header=True)

# Display the first few rows and schema to verify
df_fifa.show(truncate=True)
df_fifa.printSchema()

File 'PlayersDataset (1).csv' uploaded successfully.
+---+-----------------+---+--------------------+-----------+--------------------+-------+---------+-------------------+--------------------+------+-----+-------+------------+----------+-------+-------+------------+---------+--------+-----+---------+---------+------------------+---------+-----------+----------+--------------+-----------+----------------+-------------+-------+------------+----------+-------+---------+-----------+---------+-------------+----------+--------------+------------+-------+---------------+--------+------+-------+----+----+----+----+----+------+----+----+----+----+----+----+----+----+----+----+-------------------+----+----+----+----+----+----+----+----+----+----+----+
|_c0|             Name|Age|               Photo|Nationality|                Flag|Overall|Potential|               Club|           Club Logo| Value| Wage|Special|Acceleration|Aggression|Agility|Balance|Ball control|Composure|Crossing|Curve|Dribblin

In [ ]:
# show table
df_fifa.show(truncate = True)

# show schema
df_fifa.printSchema()

+---+-----------------+---+--------------------+-----------+--------------------+-------+---------+-------------------+--------------------+------+-----+-------+------------+----------+-------+-------+------------+---------+--------+-----+---------+---------+------------------+---------+-----------+----------+--------------+-----------+----------------+-------------+-------+------------+----------+-------+---------+-----------+---------+-------------+----------+--------------+------------+-------+---------------+--------+------+-------+----+----+----+----+----+------+----+----+----+----+----+----+----+----+----+----+-------------------+----+----+----+----+----+----+----+----+----+----+----+
|_c0|             Name|Age|               Photo|Nationality|                Flag|Overall|Potential|               Club|           Club Logo| Value| Wage|Special|Acceleration|Aggression|Agility|Balance|Ball control|Composure|Crossing|Curve|Dribbling|Finishing|Free kick accuracy|GK diving|GK handling|

In [ ]:
print(df_fifa.count())
print(len(df_fifa.columns))

17981
75


In [ ]:
# Count rows using rdd attribute
row_count = df_fifa.rdd.count()

print(f'The DataFrame has {row_count} rows.')


The DataFrame has 17981 rows.


In [ ]:
# Change the number of partitions
df_fifa = df_fifa.repartition(4)
df_fifa.rdd.getNumPartitions()

4

In [ ]:
# showing Age more than  20
subset_Age = df_fifa.filter(df_fifa["Age"] > 30)
subset_Age.show(10)

+-----+-------------+---+--------------------+-------------+--------------------+-------+---------+--------------------+--------------------+-----+----+-------+------------+----------+-------+-------+------------+---------+--------+-----+---------+---------+------------------+---------+-----------+----------+--------------+-----------+----------------+-------------+-------+------------+----------+-------+---------+-----------+---------+-------------+----------+--------------+------------+-------+---------------+--------+------+-------+----+----+----+----+----+------+----+----+----+----+----+----+----+----+----+----+-------------------+----+----+----+----+----+----+----+----+----+----+----+
|  _c0|         Name|Age|               Photo|  Nationality|                Flag|Overall|Potential|                Club|           Club Logo|Value|Wage|Special|Acceleration|Aggression|Agility|Balance|Ball control|Composure|Crossing|Curve|Dribbling|Finishing|Free kick accuracy|GK diving|GK handling|GK

In [ ]:
df_fifa.select("Name","Club").distinct().show(100)

+---------------+--------------------+
|           Name|                Club|
+---------------+--------------------+
|      C. Vargas|Atletico Nacional...|
|Simão Donatinho|Clube Atlético Pa...|
|   L. Coulibaly|          Angers SCO|
|      H. Osorio|Independiente San...|
|  M. Migliorini|            Avellino|
|    D. Petković|          FC Lorient|
|       O. Şahan|         Trabzonspor|
|      G. Donsah|             Bologna|
|      A. Sukhov|              FC Ufa|
|      A. Mawson|        Swansea City|
|      K. Kamara|New England Revol...|
|      P. McGinn|Partick Thistle F.C.|
|        I. Boye|           Örebro SK|
|         S. Old|           Morecambe|
|  M. Villasanti|           Temperley|
|       L. Nolan|  Accrington Stanley|
|     J. Joronen|          AC Horsens|
|    T. McCarron|          Finn Harps|
|       J. Terry|         Aston Villa|
|      F. Kessié|               Milan|
|     S. Ulreich|    FC Bayern Munich|
|      A. Stokes|           Hibernian|
|       G. Torje|Kardemir